In [5]:
!mv /util.py /content/

mv: cannot stat '/util.py': No such file or directory


In [7]:
import os

file_path = '/content/util.py'
if os.path.exists(file_path):
    print(f"File {file_path} is already available.")
else:
    print(f"File {file_path} not found. Please ensure it's uploaded or created.")

File /content/util.py is already available.


In [8]:
# Instalăm Unsloth pentru inferență rapidă
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=True)

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-8j441q0m/unsloth_b299b459129b4008a7b6e0aa5ee32291
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-8j441q0m/unsloth_b299b459129b4008a7b6e0aa5ee32291
  Resolved https://github.com/unslothai/unsloth.git to commit 94cc7eeaa211387cac07e6029fd1de95415ec5aa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 296.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 394.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 336.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 232.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 437.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 257.0 MB/s eta 0:00:00
 

In [9]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

# 1. Extragem doar setul de test
print("Încărcăm setul de test...")
dataset = load_dataset("alecs-vlad24/cv-resume-structuring-v2pro")
test = dataset['test']
print(f"Avem {len(test)} CV-uri pentru testare.")

# 2. Încărcăm MODELUL TĂU ANTRENAT de pe Hugging Face
# ÎNLOCUIEȘTE cu numele exact al modelului tău de pe Hugging Face
# (ex: "alecs-vlad24/cv-structuring-llm-2026-08-05_14.19")
HUB_MODEL_NAME = "alecs-vlad24/cv-structuring-llm-v2-2026-08-12_10.49"

print(f"Descărcăm modelul fine-tuned: {HUB_MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = HUB_MODEL_NAME,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Trecem modelul în modul de inferență super-rapidă
FastLanguageModel.for_inference(model)
print("Model încărcat și pregătit pentru predicții!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Încărcăm setul de test...


README.md:   0%|          | 0.00/853 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.67MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

...

Generating test split:   0%|          | 0/350 [00:00<?, ? examples/s]

Avem 350 CV-uri pentru testare.
Descărcăm modelul fine-tuned: alecs-vlad24/cv-structuring-llm-v2-2026-08-12_10.49...
==((====))==  Unsloth 2026.8.15: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model încărcat și pregătit pentru predicții!


In [11]:
import torch
from unsloth import FastLanguageModel
from transformers import set_seed
from util import evaluate

# Trecem modelul în modul de inferență
FastLanguageModel.for_inference(model)

# Definim funcția predict
def model_predict(item):
    input_messages = [m for m in item["messages"] if m["role"] != "assistant"]

    inputs = tokenizer.apply_chat_template(
        input_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs,
            max_new_tokens=1024,
            use_cache=True,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_len = inputs.shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [12]:
from transformers import set_seed

set_seed(42)

# Rulăm evaluarea efectivă pe setul de test
evaluate(model_predict, test)


🚀 Începem evaluarea corectată pe 350 CV-uri...


Procesare Inference:   0%|          | 0/350 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Procesare Inference: 100%|██████████| 350/350 [59:52<00:00, 10.27s/it]


📊 REZULTATE FINALE REALISTE: FINE-TUNED MODEL
✅ JSON Format Valid:        98.6%
🎯 Acuratețe Potrivire Rol:  85.7%
💡 Suprapunere Skill-uri:    68.5% (Jaccard Index)


In [13]:
import json
from util import parse_json_safely

# Inspectăm primul exemplu din setul de test
example = test[0]

# 1. Ce este în Ground Truth (Răspunsul corect din dataset)
gt_text = [m for m in example["messages"] if m["role"] == "assistant"][0]["content"]
gt_json = parse_json_safely(gt_text)

# 2. Ce generează Modelul
pred_text = model_predict(example)
pred_json = parse_json_safely(pred_text)

print("🔍 DEBUG GROUND TRUTH (Ce e în dataset):")
print(json.dumps(gt_json, indent=2))

print("\n🤖 DEBUG PREDICTOR (Ce a generat modelul):")
print(json.dumps(pred_json, indent=2))

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 DEBUG GROUND TRUTH (Ce e în dataset):
{
  "Candidate_Role": "Project Manager",
  "Experience_Level": "Lead",
  "Years_Of_Experience": 12,
  "Primary_Skills": [
    "ETL",
    "Data Warehousing",
    "Business Intelligence",
    "Informatica",
    "Power BI",
    "Salesforce",
    "SQL",
    "Data Modeling",
    "Data Governance",
    "Data Quality",
    "SSAS",
    "PowerCenter",
    "Data Analysis",
    "Project Management",
    "Change Management",
    "MS Office",
    "Business Objects",
    "Data Integration"
  ],
  "Clean_Summary": "The candidate has 12 years of experience in project management with a strong focus on ETL, data warehousing, and business intelligence. They have a proven track record in managing large teams and complex projects, ensuring successful delivery and performance improvement."
}

🤖 DEBUG PREDICTOR (Ce a generat modelul):
{
  "Candidate_Role": "Project Manager",
  "Experience_Level": "Senior",
  "Years_Of_Experience": 12,
  "Primary_Skills": [
    "ETL",
 